In [ ]:
"""
CDC Detector - Clock Domain Crossing Detection and Timing Analysis

This module detects CDC paths and analyzes timing violations in digital designs.
"""

from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple
from metastability_analyzer import MetastabilityAnalyzer, CDCPath, FlipFlopParameters


@dataclass
class Signal:
    """Represents a digital signal in the design"""
    name: str
    clock_domain: str
    bit_width: int = 1
    is_control: bool = False
    is_data: bool = True


@dataclass
class ClockDomain:
    """Represents a clock domain in the design"""
    name: str
    frequency: float  # Hz
    phase: float = 0.0  # radians
    is_async: bool = True


@dataclass
class CDCPathInfo:
    """Detailed information about a detected CDC path"""
    source_signal: Signal
    dest_signal: Signal
    source_domain: ClockDomain
    dest_domain: ClockDomain
    has_synchronizer: bool
    synchronizer_type: Optional[str] = None
    timing_violations: List[str] = None
    
    def __post_init__(self):
        if self.timing_violations is None:
            self.timing_violations = []


class CDCDetector:
    """
    Detects clock domain crossings and analyzes timing risks.
    
    Attributes:
        clock_domains (Dict[str, ClockDomain]): Registered clock domains
        signals (Dict[str, Signal]): Registered signals
        cdc_paths (List[CDCPathInfo]): Detected CDC paths
    """
    
    def __init__(self):
        self.clock_domains: Dict[str, ClockDomain] = {}
        self.signals: Dict[str, Signal] = {}
        self.cdc_paths: List[CDCPathInfo] = []
    
    def add_clock_domain(self, name: str, frequency: float, 
                        phase: float = 0.0, is_async: bool = True) -> ClockDomain:
        """Register a clock domain"""
        domain = ClockDomain(
            name=name,
            frequency=frequency,
            phase=phase,
            is_async=is_async
        )
        self.clock_domains[name] = domain
        return domain
    
    def add_signal(self, name: str, clock_domain: str, 
                   bit_width: int = 1, is_control: bool = False,
                   is_data: bool = True) -> Signal:
        """Register a signal in a clock domain"""
        if clock_domain not in self.clock_domains:
            raise ValueError(f"Clock domain '{clock_domain}' not found")
        
        signal = Signal(
            name=name,
            clock_domain=clock_domain,
            bit_width=bit_width,
            is_control=is_control,
            is_data=is_data
        )
        self.signals[name] = signal
        return signal
    
    def detect_cdc_path(self, source_signal_name: str, 
                       dest_signal_name: str,
                       has_synchronizer: bool,
                       synchronizer_type: str = None) -> Optional[CDCPathInfo]:
        """
        Detect and analyze a clock domain crossing.
        
        Args:
            source_signal_name: Name of source signal
            dest_signal_name: Name of destination signal
            has_synchronizer: Whether path has synchronizer
            synchronizer_type: Type of synchronizer (e.g., '2-flop', '3-flop', 'FIFO')
            
        Returns:
            CDCPathInfo if CDC detected, None if same domain
        """
        if source_signal_name not in self.signals:
            raise ValueError(f"Signal '{source_signal_name}' not found")
        if dest_signal_name not in self.signals:
            raise ValueError(f"Signal '{dest_signal_name}' not found")
        
        source = self.signals[source_signal_name]
        dest = self.signals[dest_signal_name]
        
        # Check if this is actually a CDC (different clock domains)
        if source.clock_domain == dest.clock_domain:
            return None  # Same domain, not a CDC
        
        source_domain = self.clock_domains[source.clock_domain]
        dest_domain = self.clock_domains[dest.clock_domain]
        
        # Analyze timing violations
        violations = self._analyze_timing_violations(
            source_domain, dest_domain, 
            has_synchronizer, synchronizer_type
        )
        
        cdc_info = CDCPathInfo(
            source_signal=source,
            dest_signal=dest,
            source_domain=source_domain,
            dest_domain=dest_domain,
            has_synchronizer=has_synchronizer,
            synchronizer_type=synchronizer_type,
            timing_violations=violations
        )
        
        self.cdc_paths.append(cdc_info)
        return cdc_info
    
    def _analyze_timing_violations(self, source_domain: ClockDomain,
                                  dest_domain: ClockDomain,
                                  has_synchronizer: bool,
                                  synchronizer_type: str) -> List[str]:
        """Analyze potential timing violations for a CDC path"""
        violations = []
        
        # Check if synchronizer is present
        if not has_synchronizer:
            violations.append("NO_SYNCHRONIZER")
        
        # Check synchronizer type adequacy
        if has_synchronizer:
            if synchronizer_type == '2-flop':
                # 2-flop is adequate for single-bit control signals
                pass
            elif synchronizer_type == '3-flop':
                # 3-flop provides better metastability protection
                pass
            elif synchronizer_type == 'FIFO':
                # FIFO for multi-bit data
                pass
            else:
                violations.append("UNKNOWN_SYNCHRONIZER_TYPE")
        
        # Check clock frequency ratio (high ratios are more risky)
        freq_ratio = max(source_domain.frequency, dest_domain.frequency) / \
                     min(source_domain.frequency, dest_domain.frequency)
        if freq_ratio > 10:
            violations.append("HIGH_FREQUENCY_RATIO")
        
        # Check for asynchronous domains
        if not source_domain.is_async or not dest_domain.is_async:
            # Synchronous CDC (related clocks) may need different handling
            violations.append("SYNCHRONOUS_CDC")
        
        return violations
    
    def get_cdc_risk_level(self, cdc_info: CDCPathInfo) -> str:
        """
        Determine risk level for a CDC path.
        
        Returns:
            str: 'CRITICAL', 'HIGH', 'MEDIUM', 'LOW'
        """
        if not cdc_info.has_synchronizer:
            return "CRITICAL"
        
        if "NO_SYNCHRONIZER" in cdc_info.timing_violations:
            return "CRITICAL"
        
        if "HIGH_FREQUENCY_RATIO" in cdc_info.timing_violations:
            if cdc_info.synchronizer_type in ['2-flop', '3-flop']:
                return "HIGH"
            else:
                return "MEDIUM"
        
        if len(cdc_info.timing_violations) > 0:
            return "MEDIUM"
        
        return "LOW"
    
    def detect_all_cdc_paths(self, signal_connections: List[Tuple[str, str]],
                           synchronizers: Dict[str, str] = None) -> List[CDCPathInfo]:
        """
        Detect all CDC paths from a list of signal connections.
        
        Args:
            signal_connections: List of (source, dest) signal name tuples
            synchronizers: Dict mapping connection strings to synchronizer types
            
        Returns:
            List of detected CDCPathInfo objects
        """
        if synchronizers is None:
            synchronizers = {}
        
        self.cdc_paths = []  # Reset
        
        for source_name, dest_name in signal_connections:
            conn_key = f"{source_name}->{dest_name}"
            sync_info = synchronizers.get(conn_key, {})
            
            has_sync = sync_info.get('has_synchronizer', False)
            sync_type = sync_info.get('type', None)
            
            self.detect_cdc_path(source_name, dest_name, has_sync, sync_type)
        
        return self.cdc_paths
    
    def generate_detection_report(self) -> str:
        """Generate a report of all detected CDC paths"""
        if len(self.cdc_paths) == 0:
            return "No CDC paths detected."
        
        report = []
        report.append("=" * 60)
        report.append("CDC DETECTION REPORT")
        report.append("=" * 60)
        report.append("")
        
        for i, cdc in enumerate(self.cdc_paths, 1):
            risk_level = self.get_cdc_risk_level(cdc)
            risk_emoji = {"CRITICAL": "🔴", "HIGH": "🟠", "MEDIUM": "🟡", "LOW": "🟢"}
            
            report.append(f"CDC Path #{i}")
            report.append("-" * 40)
            report.append(f"  Source: {cdc.source_signal.name} @ {cdc.source_domain.name}")
            report.append(f"  Dest:   {cdc.dest_signal.name} @ {cdc.dest_domain.name}")
            report.append(f"  Source Freq: {cdc.source_domain.frequency/1e6:.2f} MHz")
            report.append(f"  Dest Freq:   {cdc.dest_domain.frequency/1e6:.2f} MHz")
            report.append(f"  Synchronizer: {'Yes' if cdc.has_synchronizer else 'No'}")
            if cdc.synchronizer_type:
                report.append(f"  Type: {cdc.synchronizer_type}")
            report.append(f"  Risk Level: {risk_emoji.get(risk_level, '⚪')} {risk_level}")
            
            if cdc.timing_violations:
                report.append(f"  Violations: {', '.join(cdc.timing_violations)}")
            report.append("")
        
        # Summary
        critical_count = sum(1 for c in self.cdc_paths if self.get_cdc_risk_level(c) == "CRITICAL")
        high_count = sum(1 for c in self.cdc_paths if self.get_cdc_risk_level(c) == "HIGH")
        
        report.append("Summary:")
        report.append(f"  Total CDC Paths: {len(self.cdc_paths)}")
        report.append(f"  Critical Risk: {critical_count}")
        report.append(f"  High Risk: {high_count}")
        report.append("=" * 60)
        
        return "\n".join(report)